# 🎓 Mini Project: **AI Study Buddy**
### Using Hugging Face models with LangChain — *with a token, without a token, with LangChain, and without it*

In this mini project you will build a small **"Explain it to me" tutor** and run the *same* task four different ways.
The goal is not the tutor itself, but to understand **where Hugging Face ends and where LangChain begins.**

> **Why this notebook replaces the old Flan-T5 demo:** `HuggingFaceHub`, `langchain.llms`, `LLMChain` and `chain.run()` are all deprecated or removed in current LangChain. The modern pattern is the `langchain-huggingface` package + **LCEL** (the `|` pipe syntax).

## 🗺️ The Big Picture: a 2 × 2 Map

Every way of calling a Hugging Face model falls into one of these four boxes:

| | **No LangChain** (direct) | **With LangChain** |
|---|---|---|
| **No token** → model runs **locally** in this Colab | **Path A** · `transformers.pipeline` | **Path B** · `HuggingFacePipeline` + `ChatHuggingFace` |
| **With token** → model runs on **Hugging Face's servers** | **Path C** · `huggingface_hub.InferenceClient` | **Path D** · `HuggingFaceEndpoint` + `ChatHuggingFace` |

**Key idea to remember:**
* The **token** decides *where the model runs* (your machine vs. Hugging Face's cloud).
* **LangChain** decides *how you build around the model* (prompts, parsing, chaining) — it **wraps** the model, it does not replace it.
* Once wrapped, the model is just a swappable component in the chain. We will prove this by changing only **one line** to switch between Path B and Path D.

## 📋 Steps in this Project

1. Install libraries and set up the project configuration
2. Understand the two meanings of "token"
3. **Path A** — Local model, no token, no LangChain
4. **Path B** — Local model, no token, with LangChain
5. Set up the Hugging Face access token *(optional)*
6. **Path C** — Hosted model, with token, no LangChain
7. **Path D** — Hosted model, with token, with LangChain
8. Compare all four paths side by side
9. Exercises and key takeaways

## Step 1: Install Libraries and Set Up the Project

| Package | Role |
|---|---|
| `transformers` | Loads and runs models locally (PyTorch is pre-installed in Colab) |
| `huggingface_hub` | Talks to Hugging Face's hosted inference (`InferenceClient`) |
| `langchain-core` | Prompts, output parsers and the `\|` pipe syntax (LCEL) |
| `langchain-huggingface` | The bridge: wraps Hugging Face models as LangChain components |

> 💡 **Tip:** Use **GPU** (Runtime → Change runtime type → T4) for faster local generation. The notebook also works on CPU because we use a small model.

In [1]:
# Install the required libraries (-q keeps the output quiet, -U upgrades to the latest)
!pip install -q -U transformers accelerate huggingface_hub langchain-core langchain-huggingface


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip


In [3]:
import time
import torch
import pandas as pd

# ---------------- Project configuration (change models here, nowhere else) ----------------
# A small open model that runs locally, needs NO token and NO license acceptance
LOCAL_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

# A larger model served by Hugging Face's Inference Providers (needs a token).
# If this model is unavailable when you run the notebook, pick another chat model
# from https://huggingface.co/models?inference_provider=all&pipeline_tag=text-generation
HOSTED_MODEL = "Qwen/Qwen3.8-Flash-Next"

# Generation settings shared by all four paths, so the comparison is fair
MAX_NEW_TOKENS = 150   # upper limit on the length of the answer (in tokens)
DEVICE = 0 if torch.cuda.is_available() else -1   # 0 = first GPU, -1 = CPU

# ---------------- The task: an "explain it simply" tutor ----------------
SYSTEM_PROMPT = (
    "You are a friendly AI tutor. Explain concepts clearly in at most 3 sentences "
    "and use one simple analogy."
)
USER_TEMPLATE = "Explain this to a {level} learner: {topic}"   # {level} and {topic} are placeholders

LEVEL = "beginner"
TOPICS = [
    "black holes",
    "overfitting in machine learning",
    "what a neural network is",
]

print(f"Running on: {'GPU' if DEVICE == 0 else 'CPU'}")

Running on: CPU


In [4]:
HOSTED_MODEL = "Qwen/Qwen3.8-Flash-Next"

**📝 Observations**

* All model names, token limits and prompts live in **one configuration cell**. Every path below reads from it, so the comparison at the end is fair.
* `SYSTEM_PROMPT` and `USER_TEMPLATE` use `{level}` and `{topic}` placeholders. The **same template string** works in plain Python (`.format`) and in LangChain's `ChatPromptTemplate`.

## Step 2: Two Meanings of "Token" ⚠️

The word *token* causes a lot of confusion for learners. In this notebook it means **two completely different things**:

| | **Access token** (a.k.a. API key) | **Text token** |
|---|---|---|
| **What is it?** | A secret password string like `hf_xxxx...` | A small chunk of text (word, sub-word or symbol) |
| **Who uses it?** | *You*, to prove your identity to Hugging Face | *The model*, which reads and writes text only as tokens |
| **Controls** | Whether you can use Hugging Face's **hosted** models | The **length** of prompts and answers (`max_new_tokens`) |

Let's see **text tokens** in action. Loading a tokenizer needs **no access token** because the model is public.

In [7]:
from transformers import AutoTokenizer

# The tokenizer converts text <-> numbers (token IDs); no access token needed for public models
tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL)

sentence = "Hugging Face and LangChain make building AI apps easier!"

token_ids = tokenizer.encode(sentence)                 # text -> list of integers
token_strings = tokenizer.convert_ids_to_tokens(token_ids)   # integers -> readable chunks

print("Characters in sentence :", len(sentence))
print("Number of text tokens  :", len(token_ids))
print("\nToken strings:", token_strings)
print("Token IDs    :", token_ids)

Characters in sentence : 56
Number of text tokens  : 12

Token strings: ['H', 'ugging', 'ĠFace', 'Ġand', 'ĠLang', 'Chain', 'Ġmake', 'Ġbuilding', 'ĠAI', 'Ġapps', 'Ġeasier', '!']
Token IDs    : [39, 35268, 18596, 323, 22463, 18837, 1281, 4752, 15235, 10500, 8661, 0]


**📝 Observations**

* The sentence is split into **sub-word pieces**, not whole words. A leading `Ġ` marks "this piece starts with a space".
* The model never sees text, only the **integer IDs**. This is why `MAX_NEW_TOKENS = 150` limits the answer to about 110 words, not 150 words.
* From here on, **"token" = access token** unless we say *text token*.

## Step 3: Path A — Local Model · No Token · No LangChain 🟦

The most basic approach. The model is **downloaded into Colab's memory** and runs on Colab's hardware.

* ✅ No account, no key, no internet needed after download
* ✅ Full control over the model
* ❌ Limited by your hardware, so we use a small 0.5B-parameter model

In [8]:
from transformers import pipeline

# pipeline() downloads the model + tokenizer and bundles them into one callable object
local_pipe = pipeline(
    task="text-generation",
    model=LOCAL_MODEL,
    device=DEVICE,
)

def ask_direct_local(topic, level=LEVEL):
    # Chat models expect a list of messages with roles: system / user / assistant
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": USER_TEMPLATE.format(level=level, topic=topic)},  # plain Python formatting
    ]
    output = local_pipe(messages, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)  # do_sample=False -> repeatable answers

    # The pipeline returns the whole conversation; the last message is the model's reply
    return output[0]["generated_text"][-1]["content"]

print(ask_direct_local(TOPICS[0]))

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 29008.33it/s]


Black holes are like giant, dark voids in space that have so much mass that nothing, not even light, can escape from their gravitational pull. Imagine if you had a big jar of water and poured it into the center of a bowling ball - the water would be sucked out because there's no room for more! That's kind of what happens when matter falls into a supermassive black hole. The closer you get to the center, the stronger the force pulling everything towards it, making it feel like gravity is pulling on you too. It's a place where nothing can escape, but scientists still don't know all the secrets about how they form or why they exist.


**📝 Observations**

* We wrote **everything by hand**: the message list, the `.format()` call, and digging the answer out of `output[0]["generated_text"][-1]["content"]`.
* That last line is the real pain point. Every library and model returns a slightly different structure, and **LangChain exists to hide this**.
* No token was involved. The only network call was the one-time model download from the Hub.

## Step 4: Path B — Local Model · No Token · With LangChain 🟩

Same local model, but now wrapped for LangChain using **two layers**:

1. `HuggingFacePipeline` → wraps the Hugging Face pipeline as a LangChain **LLM** (text in → text out)
2. `ChatHuggingFace` → adds the model's **chat template**, so LangChain can send role-based messages (system / human)

Then we build the chain with **LCEL**:

```
prompt  |  model  |  output parser
```

Read the `|` as *"send the output of the left part into the right part"*, just like a Unix pipe.

In [9]:
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Layer 1: load the local model as a LangChain LLM
local_llm = HuggingFacePipeline.from_model_id(
    model_id=LOCAL_MODEL,
    task="text-generation",
    device=DEVICE,
    pipeline_kwargs=dict(
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        return_full_text=False,   # return only the new answer, not the prompt + answer
    ),
)

# Layer 2: wrap as a chat model so system / human roles are handled for us
local_chat_model = ChatHuggingFace(llm=local_llm)

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 26739.98it/s]
Device has 1 accelerator(s) available. Provide device={deviceId} to `from_model_id` to use availableaccelerators for execution. deviceId is -1 (default) for CPU and can be a positive integer associated with the accelerator device id.


In [10]:
# The prompt template: the same system + user text as Path A, but now reusable and validated
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", USER_TEMPLATE),
])

# A small helper that builds a chain from ANY chat model -> this is what makes models swappable
def build_chain(chat_model):
    return prompt | chat_model | StrOutputParser()   # StrOutputParser extracts the plain text answer

local_chain = build_chain(local_chat_model)

# .invoke() takes a dictionary whose keys match the {placeholders} in the template
print(local_chain.invoke({"level": LEVEL, "topic": TOPICS[0]}))

Black holes are like giant, dark voids in space that have so much mass that nothing, not even light, can escape from their gravitational pull. Imagine if you had a big jar of water and poured it into the center of a bowling ball - the water would be sucked out because there's no room for more! That's kind of what happens when matter falls into a supermassive black hole. The closer you get to the center, the stronger the force pulling everything towards it, making it feel like gravity is pulling on you too. It's a place where nothing can escape, but scientists still don't know all the secrets about how they form or why they exist.


**📝 Observations**

* The answer comes back as a **clean string**. No more `output[0]["generated_text"][-1]["content"]`; `StrOutputParser` did that job.
* The chain takes a **dictionary** (`{"level": ..., "topic": ...}`), and the prompt template fills the placeholders and raises an error if one is missing.
* `build_chain()` accepts **any** LangChain chat model. Keep this in mind for Path D.
* Still **no token**: LangChain did not change *where* the model runs. It only changed *how we talk to it*.

## Step 5: Set Up the Hugging Face Access Token 🔑 *(optional)*

Paths C and D call models hosted on Hugging Face's servers, which needs an access token. **Skip this step** if you only want to explore the local paths; the rest of the notebook adapts automatically.

### How to get a token
1. Sign in at 👉 https://huggingface.co/settings/tokens
2. Click **Create new token** → choose **Fine-grained**
3. Under permissions, tick **"Make calls to Inference Providers"** only
4. Copy the token (it starts with `hf_`) — you will not see it again

### How to store it safely in Colab
1. Click the 🔑 **Secrets** icon in Colab's left sidebar
2. **Add new secret** → Name: `HF_TOKEN`, Value: your token
3. Turn on **Notebook access**

> 🔒 **Security rules for learners**
> * **Never paste a token directly into a code cell**, because it gets saved in the notebook and leaks when you share it.
> * Use the **least privilege** principle: a token that can only call inference is safer than a `write` token (the old demo asked for `write`, which is far more than needed).
> * If a token leaks, **revoke it** from the same settings page immediately.

In [24]:
import os

HF_TOKEN = ''
try:
    # Read the token from Colab Secrets (never hard-code it in the notebook)
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    # Not in Colab, or the secret is missing -> try a normal environment variable
    HF_TOKEN = os.environ.get("HF_TOKEN")

HAS_TOKEN = bool(HF_TOKEN)

if HAS_TOKEN:
    # Many Hugging Face libraries auto-detect this environment variable.
    # We will still pass the token explicitly below so you can SEE where it is used.
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("✅ Token found. Paths C and D will run.")
else:
    print("ℹ️ No token found. Paths C and D will be skipped; Paths A and B still work.")

ℹ️ No token found. Paths C and D will be skipped; Paths A and B still work.


## Step 6: Path C — Hosted Model · With Token · No LangChain 🟧

Now the model does **not** run in Colab. We send the request over the internet to Hugging Face's **Inference Providers**, and only the answer comes back.

* ✅ Use much bigger models (7B+) without a GPU
* ✅ Nothing to download
* ❌ Needs a token, internet, and is subject to usage limits and provider availability

In [18]:
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)

# 1) Which providers serve my chosen model? (empty = no hosted API for this model)
info = api.model_info(HOSTED_MODEL, expand=["inferenceProviderMapping"])
print("Providers for", HOSTED_MODEL, "->", info.inference_provider_mapping)

# 2) If that is empty, list 10 chat models that DO have a hosted provider
for m in api.list_models(inference_provider="all", pipeline_tag="text-generation",
                         sort="downloads", limit=10):
    print(m.id)

Providers for Qwen/Qwen3.8-Flash-Next -> [InferenceProviderMapping(provider='featherless-ai', hf_model_id='Qwen/Qwen3.8-Flash-Next', provider_id='Qwen/Qwen3.8-Flash-Next', status='live', task='conversational', adapter=None, adapter_weights_path=None, type=None)]
Qwen/Qwen3-0.6B
Qwen/Qwen3-14B
deepseek-ai/DeepSeek-V3.2
Qwen/Qwen2.5-7B-Instruct
Qwen/Qwen3-8B
openai/gpt-oss-20b
Qwen/Qwen2.5-1.5B-Instruct
meta-llama/Llama-3.1-8B-Instruct
Qwen/Qwen2.5-0.5B-Instruct
Qwen/Qwen3-32B


In [ ]:
from huggingface_hub import InferenceClient

if HAS_TOKEN:
    # The token is passed here -> this is how Hugging Face knows who is calling
    hf_client = InferenceClient(model="prism-ml/Ternary-Bonsai-27B-gguf:together", token=HF_TOKEN)

    def ask_direct_hosted(topic, level=LEVEL):
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": USER_TEMPLATE.format(level=level, topic=topic)},
        ]
        response = hf_client.chat_completion(messages=messages, max_tokens=MAX_NEW_TOKENS, temperature=0.2)
        return response.choices[0].message.content   # OpenAI-style response structure

    print(ask_direct_hosted(TOPICS[0]))
else:
    print("⏭️ Skipped: no token.")

⏭️ Skipped: no token.


**📝 Observations**

* The message list is **identical** to Path A. Only the *client* changed: `pipeline(...)` → `InferenceClient(...)`.
* Notice that the parameter is `max_tokens` here but `max_new_tokens` in Path A. Every library names things slightly differently, which is another reason a wrapper like LangChain helps.
* The answer is usually **noticeably better** than Path A, since we are using a model about 14× larger (7B vs 0.5B parameters).

## Step 7: Path D — Hosted Model · With Token · With LangChain 🟥

This is the replacement for the old `HuggingFaceHub` class. We use `HuggingFaceEndpoint` to point LangChain at the hosted model, then wrap it with `ChatHuggingFace`, exactly as we did in Path B.

> 🎯 **Watch carefully:** the chain-building code does **not change at all.** We call the same `build_chain()` from Path B.

In [14]:
from langchain_huggingface import HuggingFaceEndpoint

if HAS_TOKEN:
    # LangChain wrapper for Hugging Face's hosted inference
    hosted_llm = HuggingFaceEndpoint(
        repo_id="prism-ml/Ternary-Bonsai-27B-gguf:together",
        task="text-generation",
        provider="auto",                      # let Hugging Face pick an available provider
        max_new_tokens=MAX_NEW_TOKENS,
        temperature=0.2,
        huggingfacehub_api_token=HF_TOKEN,    # the access token is passed here
    )
    hosted_chat_model = ChatHuggingFace(llm=hosted_llm)

    # THE SAME build_chain() AS PATH B -> only the model object is different
    hosted_chain = build_chain(hosted_chat_model)

    print(hosted_chain.invoke({"level": LEVEL, "topic": TOPICS[0]}))
else:
    print("⏭️ Skipped: no token.")

⏭️ Skipped: no token.


**📝 Observations**

* `build_chain(hosted_chat_model)` is the **exact same function call** as `build_chain(local_chat_model)`. The prompt, the pipe, and the parser are untouched.
* This is the real value of LangChain: **the model becomes a plug-in.** Moving from a local 0.5B model to a hosted 7B model (or to a completely different provider) is a one-line change.
* Compare Path C and Path D: Path D replaced manual message building and response parsing with `prompt | model | parser`.

## Step 8: Compare All Four Paths Side by Side 📊

We run the same topics through every available path and record the answer and the response time.

In [15]:
# Register each available path as: name -> function that takes a topic and returns an answer
backends = {
    "A · Local · Direct":       lambda topic: ask_direct_local(topic),
    "B · Local · LangChain":    lambda topic: local_chain.invoke({"level": LEVEL, "topic": topic}),
}
if HAS_TOKEN:
    backends["C · Hosted · Direct"]    = lambda topic: ask_direct_hosted(topic)
    backends["D · Hosted · LangChain"] = lambda topic: hosted_chain.invoke({"level": LEVEL, "topic": topic})

rows = []
for path_name, ask in backends.items():
    for topic in TOPICS:
        start = time.perf_counter()
        try:
            answer = ask(topic)
        except Exception as e:                      # a hosted call can fail (limits, provider down, etc.)
            answer = f"ERROR: {type(e).__name__}: {e}"
        rows.append({
            "Path": path_name,
            "Topic": topic,
            "Seconds": round(time.perf_counter() - start, 2),
            "Answer": answer.strip(),
        })

results = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", None)          # show full answers instead of cutting them off
results

,Path,Topic,Seconds,Answer
0,A · Local · Direct,black holes,3.08,"Black holes are like giant, dark voids in space that have so much mass that nothing, not even light, can escape from their gravitational pull. Imagine if you had a big jar of water and poured it into the center of a bowling ball - the water would be sucked out because there's no room for more! That's kind of what happens when matter falls into a supermassive black hole. The closer you get to the center, the stronger the force pulling everything towards it, making it feel like gravity is pulling on you too. It's a place where nothing can escape, but scientists still don't know all the secrets about how they form or why they exist."
1,A · Local · Direct,overfitting in machine learning,2.35,"Overfitting is like having a toy car that's perfect for racing but not good at driving safely on the street. It happens when your model learns too closely to the training data, which can lead to poor performance on new, unseen data. Just as you need a balance between playing with your toy car and keeping it safe, we want our models to be balanced and not overly complex. This means finding just right levels of complexity so they don't get stuck in a rut and start doing silly things (overfitting)."
2,A · Local · Direct,what a neural network is,2.21,"A neural network is like a smart computer that learns from data, just like how you learn new things by playing games or solving puzzles. It's made up of tiny parts called ""neurons"" that work together to make decisions based on the information they receive. Just as your brain uses these neurons to process information, a neural network processes data through layers of interconnected nodes (or neurons). This allows it to recognize patterns and make predictions, much like recognizing faces in a crowd!"
3,B · Local · LangChain,black holes,3.18,"Black holes are like giant, dark voids in space that have so much mass that nothing, not even light, can escape from their gravitational pull. Imagine if you had a big jar of water and poured it into the center of a bowling ball - the water would be sucked out because there's no room for more! That's kind of what happens when matter falls into a supermassive black hole. The closer you get to the center, the stronger the force pulling everything towards it, making it feel like gravity is pulling on you too. It's a place where nothing can escape, but scientists still don't know all the secrets about how they form or why they exist."
4,B · Local · LangChain,overfitting in machine learning,2.31,"Overfitting is like having a toy car that's perfect for racing but not good at driving safely on the street. It happens when your model learns too closely to the training data, which can lead to poor performance on new, unseen data. Just as you need a balance between playing with your toy car and keeping it safe, we want our models to be balanced and not overly complex. This means finding just right levels of complexity so they don't get stuck in a rut and start doing silly things (overfitting)."
5,B · Local · LangChain,what a neural network is,2.10,"A neural network is like a smart computer that learns from data, just like how you learn new things by playing games or solving puzzles. It's made up of tiny parts called ""neurons"" that work together to make decisions based on the information they receive. Just as your brain uses these neurons to process information, a neural network processes data through layers of interconnected nodes (or neurons). This allows it to recognize patterns and make predictions, much like recognizing faces in a crowd!"


In [ ]:
# Average response time per path
results.groupby("Path")["Seconds"].mean().round(2).to_frame("Avg seconds")

**📝 Observations** *(check these against your own output)*

* **A vs B** and **C vs D** should give **very similar answers**, because the model and settings are the same. LangChain changes the *plumbing*, not the *intelligence*.
* **Local vs hosted:** compare answer *quality* (hosted 7B model) with *speed*. The first local call also includes warm-up time.
* If a hosted row shows `ERROR`, read the message: it usually points to the token, rate limits, or the model not being available on any provider right now.

## Step 9: Which Path Should I Use?

| Situation | Best path |
|---|---|
| Quick experiment, privacy-sensitive data, offline work | **A** or **B** (local) |
| Need better answers than a small model can give, no GPU | **C** or **D** (hosted) |
| Just calling one model once | **A** or **C** (direct) — fewer moving parts |
| Building an app with prompts, parsers, retrieval, agents, or swappable models | **B** or **D** (LangChain) |

### 🧠 Key Takeaways
1. **Token = where the model runs.** No token → local. Token → Hugging Face's servers.
2. **LangChain = how you build around the model.** It wraps the model; it doesn't replace it.
3. `HuggingFacePipeline` (local) and `HuggingFaceEndpoint` (hosted) are interchangeable once wrapped with `ChatHuggingFace`.
4. The modern chain pattern is `prompt | model | parser` and it runs with `.invoke({...})`.
5. Keep tokens in **Colab Secrets**, give them the **minimum permissions**, and never paste them into code.

### 🔄 Migrating from the Old Demo

| Old (deprecated) | Modern replacement |
|---|---|
| `from langchain.llms import HuggingFacePipeline` | `from langchain_huggingface import HuggingFacePipeline` |
| `from langchain import HuggingFaceHub` | `from langchain_huggingface import HuggingFaceEndpoint` |
| `LLMChain(prompt=..., llm=...)` | `prompt \| model \| StrOutputParser()` |
| `llm_chain.run(question)` | `chain.invoke({"question": question})` |
| `from langchain.prompts import PromptTemplate` | `from langchain_core.prompts import PromptTemplate` (or `ChatPromptTemplate`) |
| Token role = *write* | Fine-grained token with **Inference Providers** permission only |
| `google/flan-t5-xxl` on the free serverless API | A chat/instruct model from the Inference Providers list, or any small local model |

## ✏️ Exercises

1. **Swap the model.** Change `LOCAL_MODEL` to another small instruct model from the Hub and re-run Paths A and B. What changes in the answers?
2. **Add a variable.** Extend `USER_TEMPLATE` with a `{max_sentences}` placeholder. Which of the four paths needs code changes besides the template and the input dictionary?
3. **Change the personality.** Edit `SYSTEM_PROMPT` to make the tutor a pirate. Did all four paths follow the instruction equally well? Why might the 0.5B model struggle?
4. **Token budget.** Set `MAX_NEW_TOKENS = 20`. What happens to the answers, and why? (Hint: revisit Step 2.)
5. **Stretch.** Use `.batch([...])` on `local_chain` to answer all topics in one call, and compare the time with a `for` loop.